<a id="top"></a>
<div class="list-group" id="list-tab" role="tablist">
<h1 class="list-group-item list-group-item-action active" data-toggle="list" style='background:#005097; border:0' role="tab" aria-controls="home"><center>Model Selection</center></h1>

# Model selection: a practical view

This notebook complements the lecture notes on model selection. The goal here is not to repeat the full theory, but to connect the ideas from the notes to concrete `scikit-learn` code and to the interpretation of the outputs.

As you go through the examples, keep three questions in mind:

1. Why is evaluating on the training data methodologically wrong?
2. How do `holdout`, `cross-validation`, and `nested cross-validation` appear in code?
3. How do we keep model selection separate from final evaluation?

For the formal definitions and the broader conceptual discussion, refer to Sections 1, 5, and 6 of the lecture notes.

# Testing on the training data

The next code cell makes a mistake on purpose: it trains a model on `X` and then evaluates that same model on the same `X`.

This usually yields an overly optimistic estimate, because the model is being measured on examples it has already seen during fitting.

In [2]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

iris = load_iris()

X = iris.data
y = iris.target

clf = RandomForestClassifier(n_estimators=2, random_state=0)

# X is our training data
clf.fit(X, y)

# This will result in an overly optimistic estimation since we are using X again!
y_pred = clf.predict(X)

acc = accuracy_score(y, y_pred)
print(f'Accuracy: {acc:.2f}')

Accuracy: 0.97


# Hold-out method

## Two-way holdout

The simplest fix is to separate the data into two roles:

- **training set**: used to fit the model;
- **test set**: used only to estimate performance on unseen data.

In `scikit-learn`, this is typically done with `train_test_split`. The next cell uses a single split to create a basic holdout evaluation.

In [3]:
from sklearn.model_selection import train_test_split

# split in train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

clf = RandomForestClassifier(n_estimators=2, random_state=0)
clf.fit(X_train, y_train)

# test with unseen data
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f'Accuracy: {acc:.2f}')

Accuracy: 0.91


## Three-way holdout

Two-way holdout becomes problematic as soon as the test set starts influencing modeling decisions. If we compare several candidates and keep looking at the same test set, that test set is no longer a clean final evaluation.

A common fix is to separate three roles:

- **training set**: fits model parameters;
- **validation set**: compares candidates and tunes hyperparameters;
- **test set**: is opened only once, after model selection is complete.

The lecture notes discuss this workflow in more detail, including the optimistic bias that appears when the test set is reused.

## Split proportions

There is no universally correct split ratio. The choice depends on dataset size and on how much data we can afford to reserve for validation and final testing.

For this notebook, the key point is not a specific percentage, but the separation of roles between training, validation, and testing. The detailed discussion of typical split proportions is already covered in the lecture notes.

# $k$-fold Cross-validation

Holdout has two practical weaknesses: the estimate depends a lot on one particular split, and part of the data is left unused for training in that split.

Cross-validation addresses both issues by repeating the train/validation cycle across multiple folds:

- each fold acts as validation once;
- the remaining folds act as training data;
- the final estimate comes from aggregating the fold scores.

In this notebook, the emphasis is on the computational workflow. The broader "big picture" showing where cross-validation fits relative to a final test set is discussed in Section 5.1 of the notes.

## Function `cross_val_score`

Before using the `scikit-learn` helper, the next code cell implements a minimal $k$-fold procedure manually with $k = 2$. The point is to make the rotation of folds explicit before we switch to the built-in API.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

iris = load_iris()
X = iris.data
y = iris.target

# divide the original dataset in 2 equal parts
X1, X2, y1, y2 = train_test_split(X, y, train_size=0.5, random_state=42)

# fit and evaluate first model
model = KNeighborsClassifier(n_neighbors=3)
model.fit(X1, y1)
y1_pred = model.predict(X2)

# fit and evaluate second model
model = KNeighborsClassifier(n_neighbors=3)
model.fit(X2, y2)
y2_pred = model.predict(X1)

acc_fold_1 = accuracy_score(y2, y1_pred)
acc_fold_2 = accuracy_score(y1, y2_pred)

print(f"Accuracy of fold 1: {acc_fold_1}")
print(f"Accuracy of fold 2: {acc_fold_2}")
# average accuracy of both folds
acc = (acc_fold_1 + acc_fold_2) / 2
print(f"Average accuracy: {acc}")

Accuracy of fold 1: 0.9733333333333334
Accuracy of fold 2: 0.9466666666666667
Average accuracy: 0.96


Implementing cross-validation manually becomes cumbersome as soon as $k$ grows. `scikit-learn` therefore provides [`cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html), which automates the repeated split-train-evaluate cycle.

For the purposes of this notebook, the key inputs are:
- `estimator`: the model being evaluated;
- `X`, `y`: the data and targets;
- `cv`: the number of folds;
- `scoring`: the metric used in each fold.

The output is one score per fold. That is useful because we can inspect not only the mean score, but also how much the score varies across partitions.

The next example repeats the previous experiment with `cross_val_score` and `cv=2`, so that the code stays short while the logic remains easy to compare with the manual implementation.

In [5]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=2, scoring = 'accuracy')

# print the scores for each fold
print(f'Cross-validation scores: {scores}')

# print the average and standard deviation of the scores
print(f'Average score: {scores.mean()}')
print(f'Std deviation of scores: {scores.std()}')

Cross-validation scores: [0.96 0.92]
Average score: 0.94
Std deviation of scores: 0.019999999999999962


## Function `cross_validate`

`cross_validate` plays a role similar to `cross_val_score`, but it is more informative. Didactically, it is useful because it makes the evaluation procedure look more like an explicit experiment rather than a single aggregated call.

The helper function below wraps a 3-fold evaluation so that we can compare several candidate models with less repetition.

In [6]:
from sklearn.model_selection import cross_validate
import timeit

def do_cross_validation(clf, print_model=False, print_duration=False):
    start = timeit.default_timer()
    cv = cross_validate(clf, X, y, scoring='accuracy', cv=3)
    scores = ' + '.join(f'{s:.2f}' for s in cv["test_score"])
    mean_ = cv["test_score"].mean()
    msg = f'Cross-validated accuracy: ({scores}) / 3 = {mean_:.2f}'

    if print_model:
        msg = f'\nClassifier: {clf}\n{msg}'
        print(msg)

    if print_duration:
        duration = timeit.default_timer() - start
        print(f"Duration: {duration:.2f} seconds")

In [7]:
clf = RandomForestClassifier(n_estimators=2, random_state=0)
do_cross_validation(clf, True, True)


Classifier: RandomForestClassifier(n_estimators=2, random_state=0)
Cross-validated accuracy: (0.98 + 0.92 + 0.96) / 3 = 0.95
Duration: 0.01 seconds


In [8]:
from sklearn.svm import SVC

start = timeit.default_timer()
svc = SVC(random_state=0)
print('Default value for kernel: ', svc.kernel)
do_cross_validation(svc, True, True)

Default value for kernel:  rbf

Classifier: SVC(random_state=0)
Cross-validated accuracy: (0.96 + 0.98 + 0.94) / 3 = 0.96
Duration: 0.01 seconds


The next cell uses the same evaluation routine for several candidates. This is the point where model evaluation becomes model selection: the procedure stays fixed, but the estimators change.

In [9]:
do_cross_validation(SVC(kernel='linear', random_state=0), print_model=True)
do_cross_validation(SVC(kernel='poly', random_state=0), print_model=True)
do_cross_validation(RandomForestClassifier(n_estimators=2, random_state=0), print_model=True)
do_cross_validation(RandomForestClassifier(n_estimators=5, random_state=0), print_model=True)


Classifier: SVC(kernel='linear', random_state=0)
Cross-validated accuracy: (1.00 + 1.00 + 0.98) / 3 = 0.99

Classifier: SVC(kernel='poly', random_state=0)
Cross-validated accuracy: (0.98 + 0.94 + 0.98) / 3 = 0.97

Classifier: RandomForestClassifier(n_estimators=2, random_state=0)
Cross-validated accuracy: (0.98 + 0.92 + 0.96) / 3 = 0.95

Classifier: RandomForestClassifier(n_estimators=5, random_state=0)
Cross-validated accuracy: (0.98 + 0.94 + 0.94) / 3 = 0.95


# Nested cross-validation

Once hyperparameter tuning enters the workflow, a single cross-validation loop is no longer enough if we want an unbiased estimate of performance.

Nested cross-validation separates two roles:
- the **inner loop** chooses hyperparameters;
- the **outer loop** evaluates the full selection procedure.

This separation is what prevents the same data from being used both to choose and to judge the model.

In `scikit-learn`, a useful mental mapping is:

- `GridSearchCV` implements the inner loop;
- an external call to cross-validation over that object implements the outer loop.

So the reported score refers to the procedure "search + retrain + evaluate", not just to one fixed hyperparameter setting.

In [10]:
from sklearn.model_selection import GridSearchCV

start = timeit.default_timer()
# random forest inner loop
clf_grid = GridSearchCV(RandomForestClassifier(random_state=0), param_grid={'n_estimators': [2, 5]})
# random forest outer loop
do_cross_validation(clf_grid, print_model=True, print_duration=True)

start = timeit.default_timer()
# svc inner loop
svc_grid = GridSearchCV(SVC(random_state=0), param_grid={'kernel': ['linear', 'poly']})
# svc outer loop
do_cross_validation(svc_grid, print_model=True, print_duration=True)


Classifier: GridSearchCV(estimator=RandomForestClassifier(random_state=0),
             param_grid={'n_estimators': [2, 5]})
Cross-validated accuracy: (0.98 + 0.92 + 0.96) / 3 = 0.95
Duration: 0.11 seconds

Classifier: GridSearchCV(estimator=SVC(random_state=0),
             param_grid={'kernel': ['linear', 'poly']})
Cross-validated accuracy: (1.00 + 0.94 + 0.98) / 3 = 0.97
Duration: 0.03 seconds


## Nested CV and the final model

Nested cross-validation is primarily an evaluation procedure. It gives us a less biased estimate of how well the selection strategy generalizes, but it does not directly produce a single deployable model.

After nested CV is complete, the usual next step is to refit a final model on all available training data, often by running one last hyperparameter search on that full dataset.

The important distinction is this: the number we report as estimated generalization performance still comes from the nested CV procedure, not from the final refitted model.

## Nested CV - Classification example

Read the next code cell in five steps:
1. define the outer loop;
2. define the inner loop;
3. wrap each candidate model in `GridSearchCV`;
4. use the outer loop to evaluate that search procedure;
5. only after that, refit the chosen family on the full dataset.

In [11]:
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import numpy as np

# `outer_cv` estimates generalization performance
outer_cv = KFold(3)

# `inner_cv` is used only to choose hyperparameters
inner_cv = KFold(3)

# create a synthetic classification dataset
X_cls, y_cls = make_classification(n_samples=1000, n_features=10, random_state=0)

# map model names to an estimator and its search space
models_and_parameters = {
    'svc': (SVC(), {'C': [0.01, 0.05, 0.1, 1]}),
    'rfc': (RandomForestClassifier(random_state=0), {'max_depth': [5, 10, 50, 100, 200, 500]})
}

outer_mean_scores = {}

for name, (model, params) in models_and_parameters.items():
    classifier_that_optimizes_its_hyperparams = GridSearchCV(
        estimator=model,
        param_grid=params,
        cv=inner_cv,
        scoring='accuracy'
    )

    scores_across_outer_folds = cross_val_score(
        classifier_that_optimizes_its_hyperparams,
        X_cls,
        y_cls,
        cv=outer_cv,
        scoring='accuracy'
    )

    outer_mean_scores[name] = np.mean(scores_across_outer_folds)
    error_summary = 'Model: {name}\nAccuracy in the 3 outer folds: {scores}.\nAverage acc: {avg}'
    print(error_summary.format(
        name=name,
        scores=scores_across_outer_folds,
        avg=np.mean(scores_across_outer_folds)
    ))
    print()

print('Average score across the outer folds: ', outer_mean_scores)

many_stars = '\n' + '*' * 100 + '\n'
print(many_stars + 'Now we choose the best model family and refit on the whole dataset' + many_stars)

best_model_name, best_model_avg_score = max(
    outer_mean_scores.items(),
    key=lambda name_and_score: name_and_score[1]
)

best_model, best_model_params = models_and_parameters[best_model_name]

# Final training step after evaluation: run one last search on all available data.
final_classifier = GridSearchCV(best_model, best_model_params, cv=inner_cv, scoring='accuracy')
final_classifier.fit(X_cls, y_cls)

print('Best model family: \n\t{}'.format(best_model), end='\n\n')
print('Estimated generalization performance from nested CV (accuracy):\n\t{}'.format(
    best_model_avg_score
), end='\n\n')
print('Best parameter choice after the final search on the whole dataset: \n\t{params}'
      '\n(using `{cv}` in the final refit step).'.format(
      params=final_classifier.best_params_, cv=inner_cv))

Model: svc
Accuracy in the 3 outer folds: [0.91916168 0.92192192 0.89489489].
Average acc: 0.9119928311545079

Model: rfc
Accuracy in the 3 outer folds: [0.92814371 0.91891892 0.91291291].
Average acc: 0.9199918481355608

Average score across the outer folds:  {'svc': 0.9119928311545079, 'rfc': 0.9199918481355608}

****************************************************************************************************
Now we choose the best model and refit on the whole dataset
****************************************************************************************************

Best model: 
	RandomForestClassifier()

Estimation of its generalization error (accuracy):
	0.9199918481355608

Best parameter choice for this model: 
	{'max_depth': 200}
(according to cross-validation `KFold(n_splits=3, random_state=None, shuffle=False)` on the whole dataset).


## Nested CV - Regression example I

The workflow is the same as in the classification example. The main differences are the estimator family and the metric used in the outer loop.

In [12]:
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import numpy as np

# `outer_cv` creates 3 folds for estimating generalization error
outer_cv = KFold(3)

# when we train on a certain fold, we use a second cross-validation
# split in order to choose hyperparameters
inner_cv = KFold(3)

# create some regression data
X, y = make_regression(n_samples=1000, n_features=10)

# give shorthand names to models and use those as dictionary keys mapping
# to models and parameter grids for that model
models_and_parameters = {
    'svr': (SVR(),
            {'C': [0.01, 0.05, 0.1, 1]}),
    'rfr': (RandomForestRegressor(),
           {'max_depth': [5, 10, 50, 100, 200, 500]})}

# we will collect the average of the scores on the 3 outer folds in this dictionary
# with keys given by the names of the models in `models_and_parameters`
average_scores_across_outer_folds_for_each_model = dict()

# find the model with the best generalization error
for name, (model, params) in models_and_parameters.items():
    # this object is a regressor that also happens to choose
    # its hyperparameters automatically using `inner_cv`
    regressor_that_optimizes_its_hyperparams = GridSearchCV(estimator = model, 
                                                            param_grid = params,
                                                            cv = inner_cv, 
                                                            scoring = 'neg_mean_squared_error')

    # estimate generalization error on the 3-fold splits of the data
    scores_across_outer_folds = cross_val_score(regressor_that_optimizes_its_hyperparams,
                                                X, 
                                                y, 
                                                cv = outer_cv, 
                                                scoring='neg_mean_squared_error')

    # get the mean MSE across each of outer_cv's 3 folds
    average_scores_across_outer_folds_for_each_model[name] = np.mean(scores_across_outer_folds)
    error_summary = 'Model: {name}\nMSE in the 3 outer folds: {scores}.\nAverage error: {avg}'
    print(error_summary.format(
        name=name, scores=scores_across_outer_folds,
        avg=np.mean(scores_across_outer_folds)))
    print()

print('Average score across the outer folds: ',
      average_scores_across_outer_folds_for_each_model)

many_stars = '\n' + '*' * 100 + '\n'
print(many_stars + 'Now we choose the best model and refit on the whole dataset' + many_stars)

best_model_name, best_model_avg_score = max(
    average_scores_across_outer_folds_for_each_model.items(),
    key=(lambda name_averagescore: name_averagescore[1]))

# get the best model and its associated parameter grid
best_model, best_model_params = models_and_parameters[best_model_name]

# now we refit this best model on the whole dataset so that we can start
# making predictions on other data, and now we have a reliable estimate of
# this model's generalization error and we are confident this is the best model
# among the ones we have tried
final_regressor = GridSearchCV(best_model, best_model_params, cv=inner_cv)
final_regressor.fit(X, y)

print('Best model: \n\t{}'.format(best_model), end='\n\n')
print('Estimation of its generalization error (negative mean squared error):\n\t{}'.format(
    best_model_avg_score), end='\n\n')
print('Best parameter choice for this model: \n\t{params}'
      '\n(according to cross-validation `{cv}` on the whole dataset).'.format(
      params=final_regressor.best_params_, cv=inner_cv))

Model: svr
MSE in the 3 outer folds: [-24987.69053218 -24775.62881368 -22390.21627398].
Average error: -24051.178539949808

Model: rfr
MSE in the 3 outer folds: [-4800.36404061 -5931.98646908 -4410.41832569].
Average error: -5047.589611793377

Average score across the outer folds:  {'svr': -24051.178539949808, 'rfr': -5047.589611793377}

****************************************************************************************************
Now we choose the best model and refit on the whole dataset
****************************************************************************************************

Best model: 
	RandomForestRegressor()

Estimation of its generalization error (negative mean squared error):
	-5047.589611793377

Best parameter choice for this model: 
	{'max_depth': 500}
(according to cross-validation `KFold(n_splits=3, random_state=None, shuffle=False)` on the whole dataset).


In [13]:
import sklearn
print('The scikit-learn version is {}.'.format(sklearn.__version__))

The scikit-learn version is 1.6.1.


## Nested CV - Regression example II

This second regression example uses a `Pipeline` with `StandardScaler` and `Ridge`. It is useful for showing two points at once:

- the nested-CV structure is unchanged;
- preprocessing that depends on the data should live inside the pipeline, so that each fold fits its own scaler without leakage.

In [14]:
from sklearn.datasets import make_regression
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import numpy as np

# Synthetic regression data keeps the example self-contained.
X_reg, y_reg = make_regression(
    n_samples=1000,
    n_features=8,
    noise=25,
    random_state=42
)

param_grid = {'ridge__alpha': [0.01, 0.1, 1, 10, 100]}
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
outer_scores = []

for train_idx, test_idx in outer_cv.split(X_reg, y_reg):
    X_train, X_test = X_reg[train_idx], X_reg[test_idx]
    y_train, y_test = y_reg[train_idx], y_reg[test_idx]

    pipeline = make_pipeline(StandardScaler(), Ridge())

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=5,
        scoring='neg_mean_squared_error'
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    outer_score = mean_squared_error(y_test, y_pred)
    outer_scores.append(outer_score)

print(f"Nested CV Mean Squared Error: {np.mean(outer_scores):.4f} ± {np.std(outer_scores):.4f}")


Nested CV Mean Squared Error: 0.5306 ± 0.0217


After nested CV, we still need a model to deploy. A common practical choice is to run one final hyperparameter search on the full dataset and then keep the resulting `best_estimator_`.

Notice what changes and what does not:

- the **evaluation** already happened in the nested-CV loop;
- the **training of the final model** happens now, on all available data;
- the performance number we should report still comes from nested CV.

In [15]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib

# Final search on the whole dataset after nested-CV evaluation.
final_search = GridSearchCV(
    estimator=make_pipeline(StandardScaler(), Ridge()),
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error'
)
final_search.fit(X_reg, y_reg)

final_pipeline = final_search.best_estimator_
joblib.dump(final_pipeline, 'final_model.joblib')

print(f"Best alpha found in the final search: {final_search.best_params_['ridge__alpha']}")
print("Final model trained on the whole dataset and saved as 'final_model.joblib'")

Final model trained and saved as 'final_model.joblib'


Interpretation of the final training step:

- the final search on the full dataset is used to obtain a deployable model;
- the selected hyperparameters come from that final search, not from a hand-picked placeholder value;
- the saved object is the full pipeline, including preprocessing.

This keeps the methodological story coherent: nested CV evaluates the procedure, and the final fit produces the model we would actually use.

# Final checklist

Before reporting a model-selection result, verify the following:

- the data used to choose the model were not reused to report final performance;
- the metric used in selection matches the goal of the application;
- preprocessing steps that depend on the data are inside a `Pipeline` when needed;
- if a final test set exists, it was consulted only once.

The lecture notes expand on these decisions and on when to prefer holdout, cross-validation, or nested cross-validation.

Metric choice is part of model selection, not an afterthought.

In classification, accuracy may be enough when classes and error costs are balanced, but other settings may require precision, recall, F1, or AUC.

In regression, the usual trade-off is between metrics that emphasize large errors more strongly (`MSE`, `RMSE`) and metrics that are often easier to interpret (`MAE`).

The notebook examples mostly use accuracy and mean squared error for simplicity. The lecture notes contain the broader discussion of how to align the metric with the application.

# References

1. [Model selection done right: A gentle introduction to nested cross-validation](https://ploomber.io/blog/nested-cv/).

2. [Which is the final model from Nested Cross Validation: Accuracy or Frequency?](https://datascience.stackexchange.com/questions/116311/which-is-the-final-model-from-nested-cross-validation-accuracy-or-frequency)

3. [What is the correct procedure for nested cross-validation?](https://stackoverflow.com/questions/64238730/what-is-the-correct-procedure-for-nested-cross-validation)

4. [Nested Cross Validation (Cynthia Rudin)](https://youtu.be/az60jS7MQhU?list=PLNeXFnYrCJneoY_rKtWJy833YiMrCRi5f)

5. [Nested cross-validation and selecting the best regression model - is this the right SKLearn process?](https://datascience.stackexchange.com/questions/13185/nested-cross-validation-and-selecting-the-best-regression-model-is-this-the-ri)

6. [Model evaluation, model selection, and algorithm selection in machine learning](https://sebastianraschka.com/blog/2016/model-evaluation-selection-part1.html)

7. [Nested cross validation for model selection](https://stats.stackexchange.com/questions/65128/nested-cross-validation-for-model-selection/65158#65158)

8. [Training on the full dataset after cross-validation?](https://stats.stackexchange.com/questions/11602/training-on-the-full-dataset-after-cross-validation)

9. [How to choose a predictive model after k-fold cross-validation?](https://stats.stackexchange.com/questions/52274/how-to-choose-a-predictive-model-after-k-fold-cross-validation)

10. [How to Train a Final Machine Learning Model](https://machinelearningmastery.com/train-final-machine-learning-model/)

11. [How to get from evaluation to final model](https://mindfulmodeler.substack.com/p/how-to-get-from-evaluation-to-final)